# Week 2

#### Goal
To make my DataFrame trustworthy enough to build on.

#### Objectives

1. **Set Timestamp as an Index:** currently the rows are numbered 0,1,2. This is suboptimal in a timestamp based approach, as otherwise I have to constantly refer to row IDs instead of timestamps. This will make working with pandas apparently easier.
2. **Investigate the missingness pattern:** I need to find out how the values are missing, if there's a big gap, or if the gaps are rather scattered randomly.
3. **Handle gaps differntly based on what caused them** 
- Pause gaps (large jumps between timestamps, e.g. 40+ seconds with no record): add a boolean column is_pause = True for those periods. Hint: your event messages from Week 1 already tell you exactly when pauses start and stop.
- Sensor dropouts (small isolated missing values, 1–3 seconds): fill them forward (forward-fill = carry the last known value into the gap), but cap it at 3 consecutive fills maximum.

In [1]:
from fitparse import FitFile
import pandas as pd

fit_file = FitFile('22105359350_ACTIVITY.fit')

## Defines a set with the message names which are not 'unknown'
msg_types = set (m.name for m in fit_file.messages if m.name and 'unknown' not in m.name)

print (msg_types)

## definition of a get_record  and get_print_records function which also prints the values

def get_print_records (record_type,amount=None):
    all_records = []
    for count,message in enumerate(fit_file.get_messages (record_type)):
        if amount is not None and count >= amount:
            break
        field_content = {
            field.name : field.value
            for field in message if field.name and 'unknown' not in field.name
        }
        all_records.append (field_content)
    for count,record_dict in enumerate(all_records):
    # print (record_dict)
        print (f"Message {count}")
        for key,value in record_dict.items():
            print (f"{key}:{value}")
#    return all_records

def get_records (record_type,amount=None):
    all_records = []
    for count,message in enumerate(fit_file.get_messages (record_type)):
        if amount is not None and count >= amount:
            break
        field_content = {
            field.name : field.value
            for field in message if field.name and 'unknown' not in field.name
        }
        all_records.append (field_content)
    return all_records

def find_gaps (timestamps):
        
    time_gap = []

    for count,timestamp in enumerate(timestamps):
        if count > 0:
            timestampdiff = (timestamp - timestamps[count-1]).total_seconds()
            if timestampdiff > 3:
                gap = {
                    'start_timestamp': timestamps[count-1],
                    'end_timestamp': timestamp,
                    'time_gap': timestampdiff
                }
                time_gap.append(gap)
    return time_gap

{'hrv', 'file_creator', 'training_settings', 'sport', 'file_id', 'lap', 'device_settings', 'record', 'event', 'time_in_zone', 'session', 'device_info', 'user_profile', 'activity', 'zones_target', 'gps_metadata'}


### Objective 1 Set Timestamp as an Index

Currently the rows are numbered 0,1,2. This is suboptimal in a timestamp based approach, as otherwise I have to constantly refer to row IDs instead of timestamps. This will make working with pandas apparently easier.

I will investigate the pandas documentation on how to do it an which methods are can be used. But before I want to see how the indexes look like.

In [2]:
df = pd.DataFrame (get_records('record'))
df.set_index('timestamp')

,cadence,cycle_length16,distance,enhanced_altitude,enhanced_speed,fractional_cadence,heart_rate,position_lat,position_long,power,temperature,enhanced_respiration_rate,accumulated_power
timestamp,,,,,,,,,,,,,
2026-03-08 13:24:47,73.0,0.00,4.30,111.6,4.721,0.0,108.0,597995822.0,103432794.0,11.0,31,NaN,NaN
2026-03-08 13:24:48,0.0,0.00,7.40,111.6,4.199,0.0,108.0,597995913.0,103433315.0,0.0,31,NaN,NaN
2026-03-08 13:24:49,0.0,0.00,12.60,112.2,4.385,0.0,108.0,597995841.0,103434205.0,0.0,31,NaN,NaN
2026-03-08 13:24:50,0.0,0.00,15.67,112.4,3.872,0.0,108.0,597995926.0,103434700.0,0.0,31,NaN,NaN
2026-03-08 13:24:51,0.0,8.02,18.08,112.4,2.827,0.0,108.0,597995947.0,103435167.0,0.0,31,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-08 15:25:17,0.0,0.00,41804.66,110.2,NaN,0.0,111.0,NaN,NaN,0.0,22,24.07,859224.0
2026-03-08 15:25:18,0.0,0.00,41804.66,110.2,NaN,0.0,110.0,NaN,NaN,0.0,22,24.07,859224.0
2026-03-08 15:25:19,0.0,0.00,41811.27,110.2,1.250,0.0,109.0,597997040.0,103416160.0,0.0,22,24.07,859224.0


The first thing I found out is that apparently if I run df.set_index('timestamp') the DataFrame is shown in the cell below with timestamp as an index. However if run the df.info() directly after RangeIndex seems not to be the timestamp. 

I will try the following:

1. Run df.info () in a separate cell, so that I'm sure that it's not a race condition between the set_index and the df.info()
2. If it's not a race condition I will make df= df.set_index('timestamp') to overwrite the original DataFrame and see if that works.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6708 entries, 0 to 6707
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   cadence                    6674 non-null   float64       
 1   cycle_length16             6423 non-null   float64       
 2   distance                   6708 non-null   float64       
 3   enhanced_altitude          6708 non-null   float64       
 4   enhanced_speed             6701 non-null   float64       
 5   fractional_cadence         6674 non-null   float64       
 6   heart_rate                 6700 non-null   float64       
 7   position_lat               6701 non-null   float64       
 8   position_long              6701 non-null   float64       
 9   power                      6686 non-null   float64       
 10  temperature                6708 non-null   int64         
 11  timestamp                  6708 non-null   datetime64[us]
 12  enhanced_respirat

The first approach failed I will try now the second approach

In [15]:
df = pd.DataFrame (get_records('record'))
df = df.set_index ('timestamp')
## I want to check here if the indexes are sorted on chronological order
df_index = df.index
print (len(df_index))

for count,ix in enumerate(df_index):
    ##print (count)
    if count < (len(df_index)-1):
        compare = df_index[count+1]
        delta = compare - ix
    if delta.total_seconds() <= 0:
         print ('not ok')

6708


The second approach worked. And the indexes seem to be sorted on chronological order. I checked with a for loop iterating through an array created from the indes and they are definetly ordered in chronological order. I don't know if there's a more efficient approach.

**Answer:**
There's a more efficient way, see below.

In [5]:
df = pd.DataFrame (get_records('record'))
df = df.set_index ('timestamp')

print (df.index.is_monotonic_increasing)

True


## Objective 2Investigate the missingness pattern

I need to find out how the values are missing, if there's a big gap, or if the gaps are rather scattered randomly.

**The first thing I'm going to check is how to iterate through a DataFrame or rather how to access single rows**

Findings:

- df.loc is used to access the rows.
- I used a for loop to access the single rows of the data frame to access them I needed to extract the index in a separate variable. 
- I also checked if there are any datasets where both are missing and it that's not the case



In [21]:
missing_power = []
missing_bpm = []
missing_both=[]

df.info

for row in df_index:
    ## This seems to be slow however I'll keep trying
    if pd.isna(df.loc[row]['power']):
        missing_power.append (df.loc[row].name)
    if pd.isna(df.loc[row]['heart_rate']):
        missing_bpm.append(df.loc[row].name)

for pwr in missing_power:
    for bpm in missing_bpm:
        if pwr == bpm:
            missing_both.append(pwr)

print(missing_power)
print (missing_bpm)
print (missing_both)
 

[Timestamp('2026-03-08 13:25:34'), Timestamp('2026-03-08 13:30:26'), Timestamp('2026-03-08 13:33:04'), Timestamp('2026-03-08 13:47:24'), Timestamp('2026-03-08 13:54:02'), Timestamp('2026-03-08 13:56:10'), Timestamp('2026-03-08 14:04:29'), Timestamp('2026-03-08 14:06:37'), Timestamp('2026-03-08 14:08:19'), Timestamp('2026-03-08 14:27:23'), Timestamp('2026-03-08 14:28:29'), Timestamp('2026-03-08 14:48:28'), Timestamp('2026-03-08 14:51:50'), Timestamp('2026-03-08 15:06:33'), Timestamp('2026-03-08 15:07:54'), Timestamp('2026-03-08 15:08:08'), Timestamp('2026-03-08 15:10:30'), Timestamp('2026-03-08 15:13:05'), Timestamp('2026-03-08 15:16:04'), Timestamp('2026-03-08 15:17:42'), Timestamp('2026-03-08 15:19:56'), Timestamp('2026-03-08 15:23:37')]
[Timestamp('2026-03-08 14:28:49'), Timestamp('2026-03-08 15:06:02'), Timestamp('2026-03-08 15:06:09'), Timestamp('2026-03-08 15:06:10'), Timestamp('2026-03-08 15:15:33'), Timestamp('2026-03-08 15:15:34'), Timestamp('2026-03-08 15:15:35'), Timestamp('2

In [ ]:
## This approach seems to be way more efficient 0.8 s above vs 0.0 s below

columns = ['heart_rate', 'power']
missing_heart_rate = df[df['heart_rate'].isna()].index
missing_power = df[df['power'].isna()].index
missing_both = df[df[columns].isna().all(axis=1)].index
print (missing_heart_rate)
print (missing_power)
print (missing_both)

DatetimeIndex(['2026-03-08 14:28:49', '2026-03-08 15:06:02',
               '2026-03-08 15:06:09', '2026-03-08 15:06:10',
               '2026-03-08 15:15:33', '2026-03-08 15:15:34',
               '2026-03-08 15:15:35', '2026-03-08 15:22:34'],
              dtype='datetime64[us]', name='timestamp', freq=None)
22
DatetimeIndex([], dtype='datetime64[us]', name='timestamp', freq=None)


Claude asked me to try to find a pattern between the missing power and the events of start pause or stop.

For that I need a dataframe with the events records.

In [17]:
evts = pd.DataFrame(get_records('event')).set_index('timestamp') ## This creates a DataFrame and sets the timestamp column directly as index

start_evts = evts[evts['event_type'] == 'start'] ## this creates a DataFrame which takes only the events of event_type start

same_time_all = [] ## variable to store the timestamp of the events which are the same as the missing power event

same_time = [] ## variable to store the timestamp of the start_events which are the same as the missing power event

for dttm in evts.index:
    #print (dttm)
    time_diff_seconds = 0
    all_time_diff_seconds = []
    for pwr in missing_power:
        time_difference = (pwr - dttm)
        time_diff_seconds = time_difference.total_seconds()
        all_time_diff_seconds.append(time_difference.total_seconds())
        if (time_diff_seconds) == 0:
            same_time_all.append (dttm)


for dttm in start_evts.index:
    #print (dttm)
    time_diff_seconds = 0
    all_time_diff_seconds = []
    for pwr in missing_power:
        time_difference = (pwr - dttm)
        time_diff_seconds = time_difference.total_seconds()
        all_time_diff_seconds.append(time_difference.total_seconds())
        if time_diff_seconds == 0:
            same_time.append (dttm)

print (same_time)

print (same_time_all)


[Timestamp('2026-03-08 13:25:34'), Timestamp('2026-03-08 13:30:26'), Timestamp('2026-03-08 13:33:04'), Timestamp('2026-03-08 13:47:24'), Timestamp('2026-03-08 13:54:02'), Timestamp('2026-03-08 13:56:10'), Timestamp('2026-03-08 14:04:29'), Timestamp('2026-03-08 14:06:37'), Timestamp('2026-03-08 14:08:19'), Timestamp('2026-03-08 14:27:23'), Timestamp('2026-03-08 14:28:29'), Timestamp('2026-03-08 14:48:28'), Timestamp('2026-03-08 14:51:50'), Timestamp('2026-03-08 15:06:33'), Timestamp('2026-03-08 15:07:54'), Timestamp('2026-03-08 15:08:08'), Timestamp('2026-03-08 15:10:30'), Timestamp('2026-03-08 15:13:05'), Timestamp('2026-03-08 15:16:04'), Timestamp('2026-03-08 15:17:42'), Timestamp('2026-03-08 15:19:56'), Timestamp('2026-03-08 15:23:37')]
[Timestamp('2026-03-08 13:25:34'), Timestamp('2026-03-08 13:30:26'), Timestamp('2026-03-08 13:33:04'), Timestamp('2026-03-08 13:47:24'), Timestamp('2026-03-08 13:54:02'), Timestamp('2026-03-08 13:56:10'), Timestamp('2026-03-08 14:04:29'), Timestamp('2

Again Claude knew how to do it with pandas in one line and this is the way to go

In [ ]:
import numpy as np
evts = pd.DataFrame(get_records('event')).set_index('timestamp')

start_evts = evts[evts['event_type'] == 'start']

##intersection is a function that returns all the timestamps which are found in a given array

matches = start_evts.index.intersection(missing_power)

matches_all = evts.index.intersection(missing_power)
print (evts.loc[matches_all])
# With this I confirm that the missing power is on the start events.
print (np.array_equal (matches,same_time))
# With this I confirm that the approach using pandas is the same as my for loop.


                     event  event_group event_type timer_trigger  data
timestamp                                                             
2026-03-08 13:25:34  timer          0.0      start          auto   NaN
2026-03-08 13:30:26  timer          0.0      start          auto   NaN
2026-03-08 13:33:04  timer          0.0      start          auto   NaN
2026-03-08 13:47:24  timer          0.0      start          auto   NaN
2026-03-08 13:54:02  timer          0.0      start          auto   NaN
2026-03-08 13:56:10  timer          0.0      start          auto   NaN
2026-03-08 14:04:29  timer          0.0      start          auto   NaN
2026-03-08 14:06:37  timer          0.0      start          auto   NaN
2026-03-08 14:08:19  timer          0.0      start          auto   NaN
2026-03-08 14:27:23  timer          0.0      start          auto   NaN
2026-03-08 14:28:29  timer          0.0      start          auto   NaN
2026-03-08 14:48:28  timer          0.0      start          auto   NaN
2026-0

### Objective 3 Handle gaps differntly based on what caused them

- Pause gaps (large jumps between timestamps, e.g. 40+ seconds with no record): add a boolean column is_pause = True for those periods. Hint: your event messages from Week 1 already tell you exactly when pauses start and stop.
- Sensor dropouts (small isolated missing values, 1–3 seconds): fill them forward (forward-fill = carry the last known value into the gap), but cap it at 3 consecutive fills maximum.

For Pause Gaps I would need to have a function that recognize large jumps between timestamps. I would start with more than 3 seconds, if I recognize a time_gap I would then check the events DataFrame if I find something there. 

In [10]:
records = pd.DataFrame(get_records ('record')).set_index('timestamp')

timestamps = records.index

records.info()

gaps = find_gaps (timestamps)

print(evts.loc[gaps[0]['start_timestamp']])
print(evts.loc[gaps[0]['end_timestamp']])
print(gaps[0]['time_gap'])

<class 'pandas.DataFrame'>
DatetimeIndex: 6708 entries, 2026-03-08 13:24:47 to 2026-03-08 15:25:21
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   cadence                    6674 non-null   float64
 1   cycle_length16             6423 non-null   float64
 2   distance                   6708 non-null   float64
 3   enhanced_altitude          6708 non-null   float64
 4   enhanced_speed             6701 non-null   float64
 5   fractional_cadence         6674 non-null   float64
 6   heart_rate                 6700 non-null   float64
 7   position_lat               6701 non-null   float64
 8   position_long              6701 non-null   float64
 9   power                      6686 non-null   float64
 10  temperature                6708 non-null   int64  
 11  enhanced_respiration_rate  6649 non-null   float64
 12  accumulated_power          6700 non-null   float64
dtypes: float64(12), int64(1

Here I saw that when a gap starts there is an event stop_all, and the end timestamp correlates with a start event_type. 

Now I need to add a column to the dataframe and add the isPause to the start_timestamps to recognize when a pause starts in the records data frame.

I will try it with the assign method first as I don't want to disrupt the original records

In [18]:
records_new = records.copy()

# records_new.info
records_new['is_pause']= False 

for gap in gaps:
    records_new.loc[gap['start_timestamp'],'is_pause'] = True

print(records_new.query('is_pause'))

                     cadence  cycle_length16  distance  enhanced_altitude  \
timestamp                                                                   
2026-03-08 13:24:53      0.0             0.0     18.68              112.4   
2026-03-08 13:30:00      NaN             NaN   1289.03              128.6   
2026-03-08 13:32:34      0.0             NaN   1910.54              134.2   
2026-03-08 13:47:13      0.0             0.0   6279.69              172.4   
2026-03-08 13:53:55      0.0             0.0   8536.18              187.0   
2026-03-08 13:55:46      NaN             NaN   9067.32              189.4   
2026-03-08 14:04:19      NaN             NaN  12445.96              178.8   
2026-03-08 14:06:18      0.0             0.0  12992.32              187.0   
2026-03-08 14:06:43      NaN             NaN  12996.43              186.4   
2026-03-08 14:27:48      NaN             NaN  21330.67              113.2   
2026-03-08 14:48:13      0.0             0.0  29274.35              111.0   

To fix the sensor dropouts claude recommended to use the ffill function. This fills the value with the latest record read. We use a limit of 3 for heart rate and a limit of 1 for power. 
I got requested to use forward fill to fill all the missing power and heart rate values.

In [12]:
records_filled = records.copy()

records_filled.info()
records_filled['heart_rate']=records_filled['heart_rate'].ffill(limit=3)
records_filled['power']=records_filled['power'].ffill(limit=1)

<class 'pandas.DataFrame'>
DatetimeIndex: 6708 entries, 2026-03-08 13:24:47 to 2026-03-08 15:25:21
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   cadence                    6674 non-null   float64
 1   cycle_length16             6423 non-null   float64
 2   distance                   6708 non-null   float64
 3   enhanced_altitude          6708 non-null   float64
 4   enhanced_speed             6701 non-null   float64
 5   fractional_cadence         6674 non-null   float64
 6   heart_rate                 6700 non-null   float64
 7   position_lat               6701 non-null   float64
 8   position_long              6701 non-null   float64
 9   power                      6686 non-null   float64
 10  temperature                6708 non-null   int64  
 11  enhanced_respiration_rate  6649 non-null   float64
 12  accumulated_power          6700 non-null   float64
dtypes: float64(12), int64(1

In [ ]:
## Here I will check that actually all the values got filled.
columns = ['heart_rate', 'power']
missing_heart_rate = records_filled[records_filled['heart_rate'].isna()].index
missing_power = records_filled[records_filled['power'].isna()].index
missing_both = records_filled[records_filled[columns].isna().all(axis=1)].index
print (missing_heart_rate)
print (missing_power)
print (missing_both)

DatetimeIndex([], dtype='datetime64[us]', name='timestamp', freq=None)
DatetimeIndex([], dtype='datetime64[us]', name='timestamp', freq=None)
DatetimeIndex([], dtype='datetime64[us]', name='timestamp', freq=None)


# Summary

Today I tackled 3 objectives. 

The first one was to switch the index of the DataFrame to the timestamp column. That was relatively easy, and i learnt that many of the pandas operations don't change the main dataframe, for that we need to redefine the dataframe variable. Or when creating the dataframe use the set_index() directly. 

The second one was to find the pattern in the data missingness. I iterated with for loops through the dataframe rows and stored the timestamp if the value was NaN. What really confuses me is that when I use the dataframe.info() method I see that power has only 6686 non-null values, and the array with missing_power has only 22 entries. Same thing happened using the most efficient method isna(). 

After that I got asked by Claude to check if I could find a pattern between the missing power and the events. And definetly the 22 missing powers where the same timestamp as the start event. So when starting the timer the power isn't transmited yet.

The third task was to modify the data. First was to identify gaps in the timestamps and label them with the a new column is_pause = True. For identifing the gaps I created a function called find_gaps which returns an array with objects that include the start, end timestamp, and the length of the gap. I used the gap array to include the value True on the new column is_pause which was created with all values as False. 

After that we needed to fill the sensor dropouts. For that we used the ffill function with a limit of 1 in the power column and 3 on the heart rate.